# Model Tuning: LightGBM + CV Analysis + Optuna  
 
**Цель текущего блокнота**: Оптимизация гиперпараметров и выбор наилучшей модели.  
**Следующий шаг**: Нейтрализация, ансамблирование и финальная подготовка прогнозов — в отдельном блокноте.  

Данные уже предварительно обработаны:  
- Удалены эры с пропусками - первые 209 эр  
- В Validation эмбарго 4 эры + столбец "numerai_meta_model  ""
- 737 признаков из набора "Medium"  


**ПЛАН БЛОКНОТА**
1. Загрузка данных: Train, Validation  
2. Baseline Era-Wise CV: LGBM с `standard_large_lgbm_params`  
3. Optuna CV по `corr_mean` — первый запуск (пространство вокруг baseline)  
4. Era-Wise CV с `best_params` из Optuna  
5. Второй запуск Optuna CV — уточнённое пространство вокруг лучших параметров  
6. Оценка top-2 trials: метрики, стабильность, визуализация  
7. Era-Wise CV для двух моделей из top-2 trials  
8. Финальная модель: обучение на Train, оценка на Validation  
9. Сохранение результатов  


# Install Dependencies and Import

In [ ]:
# Установка зависимостей
%pip install -q --upgrade numerapi numerai-tools optuna seaborn
# %pip install --upgrade --force-reinstall --no-cache-dir lightgbm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import random
from datetime import timedelta
import time
import warnings
from typing import Dict, List, Tuple, Optional, Any
import gc

# Numerai
from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

# Files
import json
import os
import pickle
import cloudpickle
import shutil
from tqdm import tqdm

# ML
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer

import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour
)

# Настройки отображения
pd.set_option("display.max_columns", 500)
warnings.filterwarnings("ignore")


# Встроенные графики
%matplotlib inline

# Loading Numerai Datasets  

In [ ]:
train = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/train_medium.parquet").reset_index()

In [ ]:
validation = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/validation_medium.parquet").reset_index()

In [ ]:
# Загрузка подготовленных датасетов
# train = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/train_medium.parquet").reset_index()
# validation = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/validation_medium.parquet").reset_index()

# Загрузка метаданных
with open('/content/drive/MyDrive/Colab Notebooks/dict_medium_clean.pkl', 'rb') as f:
    dict_medium = pickle.load(f)

# Список признаков
full_feature_set = list(dict_medium['all'])

# print(f"Train: {train.shape}, Eras: {train['era'].min()}-{train['era'].max()}")
# print(f"Validation: {validation.shape}, Eras: {validation['era'].min()}-{validation['era'].max()}")
# print(f"Features: {len(full_feature_set)}")

In [ ]:
# Уменьшение данных для быстрой отладки
DEBUG = True  # False — полный набор данных
era_step = 10
n_features = 10


if DEBUG:
    # Фильтр: каждая 30-я эра
    train_eras = sorted(train['era'].unique())
    selected_eras_train = train_eras[::era_step]
    train = train[train['era'].isin(selected_eras_train)].copy()
    
    val_eras = sorted(validation['era'].unique())
    selected_eras_val = val_eras[::era_step]
    validation = validation[validation['era'].isin(selected_eras_val)].copy()
    
    # Выбор первых 10 признаков
    feature_set = full_feature_set[:n_features]

    # Фильтрация столбцов
    cols_to_keep_train = ['id', 'era', 'target'] + feature_set
    cols_to_keep_val = ['id', 'era', 'target', 'meta_model'] + feature_set

    train = train[cols_to_keep_train]
    validation = validation[cols_to_keep_val]

    print(f"Используется признаков: {len(feature_set)}/{len(full_feature_set)}")
else:
    # Полный набор признаков
    feature_set = full_feature_set
    print(f"Используется признаков: {len(feature_set)}/{len(full_feature_set)}")

summary = pd.DataFrame({
    'Dataset': ['Train', 'Validation'],
    'Rows': [train.shape[0], validation.shape[0]],
    'Columns': [train.shape[1], validation.shape[1]],
    'Start Era': [train['era'].min(), validation['era'].min()],
    'End Era': [train['era'].max(), validation['era'].max()],
    'Eras': [train['era'].nunique(), validation['era'].nunique()]
})

display(summary)

In [ ]:
# STORING RESULTS & GLOBALS
model_results = {}

cols_fold_cv = ['corr_mean', 'corr_sharpe', 'corr_std', 'corr_md', 'positive_eras', 'best_iter']
cols_metrics = ['corr_mean', 'corr_sharpe', 'corr_std', 'corr_md', 'autocorr', 'feature_exposure', 'positive_eras', 'bi_mean']
cols_mmc = ['mmc_mean', 'mmc_std', 'mmc_sharpe', 'mmc_md']

results_cv = pd.DataFrame(columns=(["model_name"] + cols_metrics))
results_val = pd.DataFrame(columns=(["model_name", "time"] + cols_metrics[:-1] + cols_mmc))


SEED = 42
random.seed(SEED), np.random.seed(SEED)
N_FOLDS = 3
PURGE = 2
EMBARGO = 2
ESR = 100

FEATURES = feature_set
TARGET = 'target'
PREDICTION = 'prediction'

# Functions

## Metrics

In [ ]:
def compute_metrics_corr(df: pd.DataFrame, pred_col: str = 'prediction', target_col: str = 'target') -> dict:

    per_era_corr = df.groupby('era').apply(lambda g: numerai_corr(g[[pred_col]], g[target_col]))

    corr_mean = per_era_corr.mean() 
    corr_std = per_era_corr.std(ddof=0)
    corr_sharpe = corr_mean / corr_std
    corr_md = (per_era_corr.cumsum().expanding(min_periods=1).max() - per_era_corr.cumsum()).max()
    positive_eras = (per_era_corr > 0).mean()

    metrics_corr = {
        'corr_mean': corr_mean,
        'corr_std': corr_std,
        'corr_sharpe': corr_sharpe,
        'corr_md': corr_md,
        'positive_eras': positive_eras,
        # "n_eras": len(per_era_corr)
    }

    return {k: float(v) for k, v in metrics_corr.items()}


In [ ]:
def compute_metrics_mmc(df: pd.DataFrame, pred_col: str = 'prediction', meta_col: str = 'meta_model', target_col: str = 'target') -> dict:

    # Удаляем NaN и группируем по эрам
    df = df.dropna(subset=[pred_col, meta_col, target_col, 'era'])
    
    per_era_mmc = df.groupby('era').apply(lambda g: correlation_contribution(g[[pred_col]], g[meta_col], g[target_col]))

    mmc_mean = per_era_mmc.mean()
    mmc_std = per_era_mmc.std(ddof=0)
    mmc_sharpe = mmc_mean / mmc_std  # if mmc_std != 0 else 0.0
    mmc_md = ((per_era_mmc.cumsum().expanding(min_periods=1).max() - per_era_mmc.cumsum()).max())

    metrics_mmc = {
        'mmc_mean': mmc_mean,
        'mmc_std': mmc_std,
        'mmc_sharpe': mmc_sharpe,
        'mmc_md': mmc_md,
    }

    return {k: float(v) for k, v in metrics_mmc.items()}


In [ ]:
def compute_autocorr(df: pd.DataFrame, pred_col: str = 'prediction') -> float:
    """
    Вычисляет автокорреляцию предсказаний по эрам: корреляция между pred[era_t] и pred[era_{t+1}].
    Возвращает: автокорреляция с лагом 1 (по эрам), или NaN если невозможно посчитать
    '0.05-0.2' – Хороший баланс стабильности и адаптивности 
    """
    pred_by_era = df.groupby('era')[pred_col].mean().sort_index()  # Агрегируем предсказания по эре
    corr = pred_by_era.corr(pred_by_era.shift(1), method='spearman')  # Сдвиг: era_t vs era_{t+1}  
    
    return corr if pd.notna(corr) else np.nan

# autocorr = compute_autocorr(oof)
# print(f"Autocorrelation (lag-1): {autocorr}")

In [ ]:
def compute_feature_exposure(df: pd.DataFrame, features_df: pd.DataFrame, pred_col: str = 'prediction') -> float:

    pred = df[pred_col].rank(pct=True, method="first")  # Получаем ранжированные предсказания (Series)
    feats = features_df.rank(pct=True, method="first")  # Ранжируем признаки (DataFrame)
    corrs = feats.corrwith(pred, method='pearson')  # Корреляция каждого признака с предсказаниями
    
    return float(corrs.abs().mean())  # Среднее абсолютных корреляций

# final_feature_exposure = compute_feature_exposure(df=oof, features_df=train[FEATURES], pred_col=PREDICTION)
# final_feature_exposure

In [ ]:
# Plot: CV Metrics per Fold
def plot_cv_metrics(fold_results_df: dict, cols_fold_cv, title: str = "CV Metrics per Fold"):
    
    df_plot = fold_results_df.iloc[:-1].copy()  # Удаляем агрегированную строку
    df_plot['fold'] = df_plot['fold'].astype(int)  # Преобразуем столбец 'fold' в int
    df_plot = df_plot.set_index('fold')  # Устанавливаем 'fold' как индекс
    

    axes = df_plot[cols_fold_cv].plot.bar(
        subplots=True,
        figsize=(14, 3),
        layout=(1, len(cols_fold_cv)),
        sharex=False,
        color="purple",
        legend=False,
        alpha=0.5,
        title=""
    )

    for ax in axes.flat:
        ax.grid(True, axis='y', alpha=0.5, linestyle='--')
        ax.set_xlabel("")
        ax.set_yticklabels([]) 

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

# plot_cv_metrics(fold_results_df, cols_fold_cv)


In [ ]:
# Plot: Correlation by Era
def plot_correlation_by_era(df: pd.DataFrame, pred_col: str = "prediction", target_col: str = "target"):

    # Вычисляем корреляцию по эрам
    per_era_corr = df.groupby("era").apply(lambda x: numerai_corr(x[[pred_col]], x[target_col]))[pred_col]
    mean_corr = per_era_corr.mean()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # === Validation CORR by Era ===
    ax1 = axes[0]
    ax1.bar(range(len(per_era_corr)), per_era_corr.values, color="darkorange", alpha=0.7)
    ax1.axhline(mean_corr, color="black", linestyle="--", linewidth=0.5,
                label=f"Mean = {mean_corr:.3f}")
    ax1.axhline(0, color="darkorange", linewidth=0.8)
    ax1.set_title("CORR by Era", fontsize=10, fontweight="bold")
    ax1.set_xlabel("Eras")
    ax1.legend()
    ax1.grid(alpha=0.3)

    # === Cumulative CORR ===
    ax2 = axes[1]
    cumcorr = per_era_corr.cumsum()
    ax2.plot(range(len(cumcorr)), cumcorr.values, color="darkorange", linewidth=2, label="Cumulative CORR")
    ax2.set_title("Cumulative CORR by Era", fontsize=10, fontweight="bold")
    ax2.set_xlabel("Eras")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# plot_correlation_by_era(oof, pred_col="prediction")

In [ ]:
def plot_mmc_by_era(
    df: pd.DataFrame,
    pred_col: str = "prediction",
    meta_col: str = "meta_model",
    target_col: str = "target"
):

    df = df.dropna(subset=[pred_col, meta_col, target_col, 'era'])

    per_era_mmc = df.groupby('era').apply(lambda g: correlation_contribution(g[[pred_col]], g[meta_col], g[target_col]))[pred_col]
    mean_mmc = per_era_mmc.mean()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # === MMC by Era ===
    ax1 = axes[0]
    ax1.bar(range(len(per_era_mmc)), per_era_mmc.values, color='purple', alpha=0.7)
    ax1.axhline(mean_mmc, color='black', linestyle='--', linewidth=0.8, label=f"Mean MMC = {mean_mmc:.4f}")
    ax1.axhline(0, color='black', linewidth=0.5)
    ax1.set_title("MMC by Era", fontsize=10, fontweight="bold")
    ax1.set_xlabel("Eras")
    ax1.set_ylabel("MMC")
    ax1.legend()
    ax1.grid(alpha=0.3)

    # === Cumulative MMC ===
    ax2 = axes[1]
    cummmc = per_era_mmc.cumsum()
    ax2.plot(range(len(cummmc)), cummmc.values, color='purple', linewidth=2, label="Cumulative MMC")
    ax2.set_title("Cumulative MMC by Era", fontsize=10, fontweight="bold")
    ax2.set_xlabel("Eras")
    ax2.set_ylabel("Cumulative MMC")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# plot_mmc_by_era(validation, pred_col="prediction", meta_col="meta_model", target_col="target")

In [ ]:
# Plot: Mean Feature Importance
def plot_feature_importance(importance_df: pd.DataFrame, top_k: int = 20):

    # Выбираем топ-k признаков и разворачиваем для отображения
    top_imp = importance_df.head(top_k).iloc[::-1]

    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.barh(top_imp["feature"], top_imp["importance"], color="steelblue", edgecolor="black", height=0.7)

    ax.set_title("Mean Feature Importance (Gain)", fontsize=14, fontweight="bold")#, pad=15)
    ax.set_ylabel("Feature")
    ax.grid(axis="x", alpha=0.5)

    plt.tight_layout()
    plt.show()

# plot_feature_importance(importance_df, top_k=20)

In [ ]:
# Plot: Prediction Distributions & Scatter
def plot_prediction_target(
    df: pd.DataFrame,
    target_col: str = "target",
    pred_col: str = "prediction",
    bins: int = 50,
    scatter_alpha: float = 0.4,
    scatter_size: int = 5,
    color_pred: str = "blue",
    color_target: str = "red",
    color_scatter: str = "purple"
):

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # === Distributions Prediction & Target ===
    ax1 = axes[0]
    sns.histplot(
        df[pred_col], bins=bins, kde=True, ax=ax1, alpha=0.6,
        label=f"Prediction", color=color_pred, stat="density"
    )
    sns.histplot(
        df[target_col], bins=bins, kde=True, ax=ax1, alpha=0.6,
        label=f"Target", color=color_target, stat="density"
    )
    # ax1.set_title(": Distribution")
    ax1.set_xlabel("Value")
    ax1.set_ylabel("Density")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # === Scatter Plot Prediction vs Target ===
    ax2 = axes[1]
    ax2.scatter(
        df[target_col], df[pred_col],
        alpha=scatter_alpha, s=scatter_size, color=color_scatter, edgecolor=None
    )
    ax2.set_xlabel("Target")
    ax2.set_ylabel("Prediction")
    # ax2.set_title("Prediction vs Target: Scatter Plot")
    ax2.grid(True, alpha=0.3)

    lims = [
        np.min([ax2.get_xlim(), ax2.get_ylim()]),
        np.max([ax2.get_xlim(), ax2.get_ylim()]),
    ]
    ax2.plot(lims, lims, linestyle="--", color="black", alpha=0.5, label="y=x")
    ax2.legend(loc="lower right")

    fig.suptitle("Prediction vs Target", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

# plot_prediction_distributions(df=oof, target_col="target", pred_col="prediction")


## CV

In [ ]:
class PurgedGroupTimeSeriesSplit:
    """
    Time series cross-validator with purged and embargoed splits.
    Splits data by eras to prevent leakage. Removes recent training eras (purge)
    and skips eras near validation (embargo) to avoid target overlap.
    """
    def __init__(self, n_splits, purge, embargo):
        self.n_splits = n_splits
        self.purge = purge
        self.embargo = embargo

    def split(self, X, y=None, eras=None):
        unique_eras = np.unique(eras)
        era_to_idx = {era: i for i, era in enumerate(unique_eras)}
        eras = np.array([era_to_idx[era] for era in eras])

        splits = np.array_split(np.arange(len(unique_eras)), self.n_splits + 1)  # Split eras into n_splits + 1 chunks
        
        for i in range(1, self.n_splits + 1):
            train_era_indices = np.concatenate(splits[:i])  # Train: all era blocks before i
            val_era_indices = splits[i]  # Val: i-th block

            # Purge: remove last `purge` eras from train
            max_train_era_idx = train_era_indices.max()
            purged_max = max_train_era_idx - self.purge

            # Embargo: skip `embargo` eras after purged_max
            min_val_era_idx = purged_max + 1 + self.embargo

            # Filter val eras to only those after embargo
            val_era_indices = val_era_indices[val_era_indices >= min_val_era_idx]
            if len(val_era_indices) == 0:
                continue  # Skip fold if no valid val eras

            # Create boolean masks
            train_mask = eras <= purged_max
            val_mask = np.isin(eras, val_era_indices)

            train_idx = np.where(train_mask)[0]
            val_idx = np.where(val_mask)[0]

            yield train_idx, val_idx


In [ ]:
def era_wise_cv(
    model_name: str,
    params: Dict[str, Any],
    train_df: pd.DataFrame,
    feature_set: List[str] = FEATURES,
    target_col: str = TARGET,
    pred_col: str = PREDICTION,
    n_folds: int = N_FOLDS,
    purge: int = PURGE,
    embargo: int = EMBARGO,
    esr: int = ESR,
) -> Dict[str, Any]:
 
    X, y = train_df[feature_set], train_df[target_col]
    eras = train_df['era'].values

    fold_results: List[Dict] = []
    all_importances: List[pd.Series] = []
    
    oof = train_df[["id", "era", target_col]].copy()
    oof[pred_col] = 0.0

    for fold, (train_idx, val_idx) in enumerate(PurgedGroupTimeSeriesSplit(n_folds, purge, embargo).split(X, y, eras)):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        eras_train, eras_val = eras[train_idx], eras[val_idx]


        model = lgb.LGBMRegressor(**params, random_state=SEED)
        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            eval_names=['train', 'val'],
            eval_metric='l2',
            callbacks=[
                lgb.early_stopping(stopping_rounds=esr, verbose=False),
                lgb.log_evaluation(period=0)
            ],
        )

        # OOF   
        val_pred = model.predict(X_val)
        row_indices = train_df.index[val_idx]
        oof.loc[row_indices, pred_col] = val_pred

        # === Compute Fold Metrics ===
        metrics = compute_metrics_corr(oof.loc[row_indices])
        best_iter = getattr(model, 'best_iteration_', None)

        fold_results.append({
            'fold': fold + 1,
            'train_eras': f'{eras_train[0]}-{eras_train[-1]}',
            'val_eras': f'{eras_val[0]}-{eras_val[-1]}',
            'corr_mean': metrics['corr_mean'],
            'corr_std': metrics['corr_std'],
            'corr_sharpe': metrics['corr_sharpe'],
            'corr_md': metrics['corr_md'],
            'positive_eras': metrics['positive_eras'],
            'best_iter': best_iter,
        })
    
        # === Feature Importance per Fold ===
        importances = model.booster_.feature_importance(importance_type='gain')
        importance_series = pd.Series(importances, index=X_train.columns, name="gain")
        all_importances.append(importance_series)


    # === Aggregate Fold Results ===
    fold_results_df = pd.DataFrame(fold_results)
    num_cols = ['corr_mean', 'corr_std', 'corr_sharpe', 'corr_md', 'positive_eras']
    mean_metrics = fold_results_df[num_cols].mean().round(4)
    bi_mean = fold_results_df.loc[fold_results_df['fold'] > 2, 'best_iter'].mean()
    bi_mean = int(round(bi_mean)) if not pd.isna(bi_mean) else None

    fold_results_df.loc[n_folds] = {**mean_metrics, 'best_iter': bi_mean, 'fold': 'mean', 'train_eras': '-', 'val_eras': '-'}
    fold_results_df[num_cols] = fold_results_df[num_cols].round(4)

    # === Overall OOF Metrics ===
    final_metrics = compute_metrics_corr(oof, pred_col=pred_col, target_col=target_col)
    final_autocorr = compute_autocorr(oof, pred_col=pred_col)
    final_feature_exposure = compute_feature_exposure(oof, train[FEATURES], pred_col=pred_col)

    # === Average Feature Importances ===
    importance_df = pd.concat(all_importances, axis=1).mean(axis=1).sort_values(ascending=False).reset_index()
    importance_df.columns = ['feature', 'importance']

    # === Final CV Summary ===
    cv_summary = {
        'model_name': model_name,
        "corr_mean": final_metrics['corr_mean'],
        'corr_std': final_metrics['corr_std'],
        'corr_sharpe': final_metrics['corr_sharpe'],
        'corr_md': final_metrics['corr_md'],
        'positive_eras': final_metrics['positive_eras'],
        'bi_mean': bi_mean,
        'autocorr': float(final_autocorr),
        'feature_exposure': final_feature_exposure,
        '---': '---',
        **params,
    }

    result = {
        'model_name': model_name,
        'params': params, 
        'cv_summary': cv_summary,
        # 'bi_mean': bi_mean,
        'fold_results_df': fold_results_df, 
        'oof': oof, 
        'importance_df': importance_df
    }

    return result

In [ ]:
def era_wise_cv_optuna(
    params: Dict[str, Any],
    train_df: pd.DataFrame,
    trial=None,
    feature_set: List[str] = FEATURES,
    target_col: str = TARGET,
    pred_col: str = PREDICTION,
    n_folds: int = N_FOLDS,
    purge: int = PURGE,
    embargo: int = EMBARGO,
    esr: int = ESR,
) -> float:

    X, y = train_df[feature_set], train_df[target_col]
    eras = train_df['era'].values

    oof = np.zeros(len(train_df))
    best_iters = []
    
    
    for fold, (train_idx, val_idx) in enumerate(PurgedGroupTimeSeriesSplit(n_folds, purge, embargo).split(X, y, eras)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(**params, random_state=SEED)
        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            eval_names=['train', 'val'],
            eval_metric='l2',
            callbacks=[
                lgb.early_stopping(stopping_rounds=esr, verbose=False),
                lgb.log_evaluation(period=0)
            ],
        )
        
        # Валидационные предсказания
        val_pred = model.predict(X_val)
        oof[val_idx] = val_pred

        # Сохраняем лучшую итерацию
        best_iter = getattr(model, 'best_iteration_', None)
        best_iters.append(best_iter)

    # Создаём временный DataFrame для вычисления корреляции по эрам
    oof_df = pd.DataFrame({
        'era': eras,
        target_col: train_df[target_col].values,
        pred_col: oof
    })
    
    # Вычисляем corr_mean по OOF
    metrics = compute_metrics_corr(oof_df, pred_col=pred_col, target_col=target_col)
    corr_mean = metrics['corr_mean']

    # Сохраняем среднюю лучшую итерацию
    bi_mean = int(np.mean(best_iters[2:]))
    trial.set_user_attr("bi_mean", bi_mean)

    return corr_mean

In [ ]:
# def era_wise_cv_run(
#     train_df: pd.DataFrame,
#     params: Dict[str, Any],
#     # model_name: str,
#     feature_set: List[str] = FEATURES,
#     target_col: str = TARGET,
#     pred_col: str = PREDICTION,
#     # n_folds: int = N_FOLDS,
#     # purge: int = PURGE,
#     # embargo: int = EMBARGO,
#     esr: int = ESR,
#     # random_state: Optional[int] = SEED,
# ) -> Tuple[Dict, pd.DataFrame, pd.DataFrame]:
    
#     X, y = train_df[feature_set], train_df[target_col]
#     eras = train_df['era'].values
    
#     fold_results: List[Dict] = []
#     all_importances: List[pd.Series] = []
    
#     oof = train_df[["id", "era", TARGET]].copy()
#     oof[pred_col] = 0.0
    

#     for fold, (train_idx, val_idx) in enumerate(PurgedGroupTimeSeriesSplit().split(X, y, eras)):

#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         train_eras = eras[train_idx]
#         val_eras = eras[val_idx]
#         train_era_start, train_era_end = train_eras[0], train_eras[-1]
#         val_era_start, val_era_end = val_eras[0], val_eras[-1]

#         model = lgb.LGBMRegressor(**params, random_state=SEED)

#         model.fit(
#             X_train, y_train,
#             eval_set=[(X_train, y_train), (X_val, y_val)],
#             eval_names=['train', 'val'],
#             eval_metric='l2',
#             callbacks=[
#                 lgb.early_stopping(stopping_rounds=esr, verbose=False),
#                 lgb.log_evaluation(period=0)
#             ],
#         )

#         val_pred = model.predict(X_val)
   
#         # OOF
#         row_indices = train_df.index[val_idx]
#         oof.loc[row_indices, pred_col] = val_pred
#         oof_val = oof.loc[row_indices].copy()

#         metrics_corr = compute_metrics_corr(oof_val)
#         best_iter = getattr(model, 'best_iteration_', None)

#         # Важность признаков
#         importances = model.booster_.feature_importance(importance_type='gain')
#         importance_series = pd.Series(importances, index=X_train.columns, name="gain")
#         all_importances.append(importance_series)

#         fold_results.append({
#             'fold': fold + 1,
#             'corr_mean': metrics_corr['corr_mean'],
#             'corr_std': metrics_corr['corr_std'],
#             'corr_sharpe': metrics_corr['corr_sharpe'],
#             'corr_md': metrics_corr['corr_md'],
#             'positive_eras': metrics_corr['positive_eras'],
#             'train_eras': f'{train_era_start}-{train_era_end}',
#             'val_eras': f'{val_era_start}-{val_era_end}',
#             'best_iter': best_iter,
#         })

#     # Преобразуем в DataFrame
#     fold_results_df = pd.DataFrame(fold_results).copy()

#     # Расчёт средних метрик и доавление в 'fold_results'
#     num_cols = ['corr_mean', 'corr_std', 'corr_sharpe', 'corr_md', 'positive_eras']
#     mean_series = fold_results_df[num_cols].mean().round(4)
#     mean_series['best_iter'] = fold_results_df.loc[fold_results_df['fold'] > 2, 'best_iter'].mean()
#     mean_series['best_iter'] = int(round(mean_series['best_iter'])) if not pd.isna(mean_series['best_iter']) else None

#     fold_results_df.loc['mean'] = mean_series
#     fold_results_df[num_cols] = fold_results_df[num_cols].round(4)
#     fold_results_df['best_iter'] = fold_results_df['best_iter'].astype('Int64')  # красиво с NaN

#     return fold_results_df, oof, all_importances


In [ ]:
# def era_wise_cv_agg(
#     df: pd.DataFrame,
#     fold_results_df: pd.DataFrame,
#     all_importances: List[pd.DataFrame],
#     model_name: str,
#     params: Dict[str, Any],
#     features_df: pd.DataFrame,
# ) -> Tuple[pd.DataFrame, pd.DataFrame, List[pd.Series]]:
    
#     final_metrics = compute_metrics_corr(df, pred_col=PREDICTION, target_col=TARGET)
#     final_autocorr = compute_autocorr(df, pred_col=PREDICTION)
#     final_feature_exposure = compute_feature_exposure(df, features_df, pred_col=PREDICTION)
#     bi_mean = fold_results_df.loc['mean', 'best_iter']

#     # Важность признаков — среднее по фолдам
#     importance_df = pd.concat(all_importances, axis=1)  # shape: (n_features, n_folds)
#     importance_df = importance_df.mean(axis=1).sort_values(ascending=False).reset_index()
#     importance_df.columns = ['feature', 'importance']

#     cv_summary = {
#         'model_name': model_name,
#         "corr_mean": final_metrics['corr_mean'],
#         'corr_std': final_metrics['corr_std'],
#         'corr_sharpe': final_metrics['corr_sharpe'],
#         'corr_md': final_metrics['corr_md'],
#         'autocorr': float(final_autocorr),
#         'feature_exposure': final_feature_exposure,
#         'positive_eras': final_metrics['positive_eras'],
#         'bi_mean': bi_mean,
#         '---': '---',
#         **params,
#     }

#     return cv_summary, fold_results_df, df, importance_df

## Save

In [ ]:
def save_model_result(
    model_results: Dict[str, Any],
    result: Dict[str, Any],
    save_to_disk: bool = True,
    save_path: str = "/content/drive/MyDrive/Colab Notebooks/Data/model_results.pkl",
) -> None:

    # Добавляем результат в словарь
    model_results[result['model_name']] = {
        'params': result['params'],
        'cv_summary': result['cv_summary'],
        'fold_results_df': result['fold_results_df'].copy(),
        'bi_mean': result['bi_mean'],
        'oof': result['oof'].copy(),
        'importance_df': result['importance_df'].copy(),
    }

    if save_to_disk:
        with open(save_path, 'wb') as f:
            pickle.dump(model_results, f, protocol=pickle.HIGHEST_PROTOCOL)

# Baseline Model with Large Params Numerai  

In [ ]:
model_name = "baseline_cv"

standard_large_lgbm_params = {
    "n_estimators": 20000,
    "learning_rate": 0.001,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "verbosity": -1,
    "device_type": "gpu",
}

result = era_wise_cv(
        model_name=model_name,
        params=standard_large_lgbm_params,
        train_df=train,
        )

# save_model_result(model_results, result)
display(result['fold_results_df'])
plot_cv_metrics(result['fold_results_df'], cols_fold_cv=cols_fold_cv)
# plot_correlation_by_era(result['oof'], target_col=TARGET, pred_col=PREDICTION)
# plot_prediction_target(result['oof'])
# plot_feature_importance(result['importance_df'])
results_cv = pd.concat([results_cv, pd.DataFrame([result['cv_summary']])], ignore_index=True, sort=False).round(4)
display(results_cv)

# Optuna 1

In [ ]:
# STUDY 1
# optuna.logging.set_verbosity(optuna.logging.WARNING)
baseline_score = result['cv_summary']['corr_mean']  # Baseline corr_mean on OOF

def objective(trial: Trial):

    params_optuna = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 300, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 64),
        # 'max_depth': max_depth,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.7),
        # 'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        # 'subsample_freq': trial.suggest_int('subsample_freq', 1, 5),
        # 'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0) ,
        # 'min_child_samples' : trial.suggest_int('min_child_samples', 20, 100)
    }

    score = era_wise_cv_optuna(params=params_optuna, train_df=train, trial=trial)

    return score


# Creating Study
study1 = optuna.create_study(direction='maximize', study_name='baseline_optuna')
study1.enqueue_trial(standard_large_lgbm_params)  # Добавляем baseline как первый trial
study1.set_user_attr('cv_score', baseline_score)  # Сохраняем baseline_score в study attributes

# Starting Optimization
study1.optimize(objective, n_trials=4, show_progress_bar=True)

# Study Results
best_trial = study1.best_trial
best_params = best_trial.params
bi_mean = best_trial.user_attrs.get('bi_mean')
trials_df = study1.trials_dataframe().sort_values('value', ascending=False).round(4)

In [ ]:
print(f"Baseline corr_mean: {baseline_score:.4f}")
print(f"Best Optuna corr_mean: {best_trial.value:.4f}")
print(f"Improvement: {best_trial.value - baseline_score:.4f}")
print("bi_mean:", bi_mean)
display(trials_df[['number', 'value', 'user_attrs_bi_mean'] + [c for c in trials_df.columns if "params_" in c]].head(10))
# plot_param_importances(study1)
# plot_optimization_history(study1)
# plot_contour(study1, params=["learning_rate", "num_leaves"])
plt.show()

In [ ]:
# MODEL WITH BEST PARAMS FOR OPTUNA 1
model_name = "tuned_1"

# Max Depth for Log
estimated_max_depth = int(np.ceil(np.log2(best_params['num_leaves']))) + 1

params_tuned_1 = best_params.copy()
params_tuned_1.update({
    'n_estimators': int(1.2 * (study1.best_trial.user_attrs.get('bi_mean'))),
    "device_type": "gpu",
    'verbosity': -1,
})

result = era_wise_cv(
        model_name=model_name,
        params=params_tuned_1,
        train_df=train,
        )

# save_model_result(model_results, result)
print(f"Estimated max_depth: ~{estimated_max_depth} (from num_leaves={best_params['num_leaves']})")
display(result['fold_results_df'])
plot_cv_metrics(result['fold_results_df'], cols_fold_cv=cols_fold_cv)
# plot_correlation_by_era(result['oof'], target_col=TARGET, pred_col=PREDICTION)
# plot_prediction_target(result['oof'])
# plot_feature_importance(result['importance_df'])
results_cv = pd.concat([results_cv, pd.DataFrame([result['cv_summary']])], ignore_index=True, sort=False).round(4)
display(results_cv)

# Opuna 2

In [ ]:
# optuna.logging.set_verbosity(optuna.logging.WARNING)
tuned_score_1 = result['cv_summary']['corr_mean']


def objective(trial: Trial):

    params_optuna = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.06, 0.07, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 64),
        # 'max_depth': max_depth,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.5),
        # 'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        # 'subsample_freq': trial.suggest_int('subsample_freq', 1, 5),
        # 'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0) ,
        # 'min_child_samples' : trial.suggest_int('min_child_samples', 20, 100)
    }

    score = era_wise_cv_optuna(params=params_optuna, train_df=train, trial=trial)

    return score


# Создаём study
study2 = optuna.create_study(direction='maximize', study_name='tuned_2')
study2.enqueue_trial(params_tuned_1)  # Добавляем baseline как первый trial
study2.set_user_attr('cv_score', tuned_score_1)  # Сохраняем baseline_score в study attributes

# Запускаем оптимизацию
study2.optimize(objective, n_trials=4, show_progress_bar=True)

# Results
best_trial = study2.best_trial
# best_params = best_trial.params
bi_mean = best_trial.user_attrs.get('bi_mean')
trials_df_2 = study2.trials_dataframe().sort_values('value', ascending=False).round(4)

print(f"Baseline corr_mean: {tuned_score_1:.4f}")
print(f"Best Optuna corr_mean: {best_trial.value:.4f}")
print(f"Improvement: {best_trial.value - tuned_score_1:.4f}")
# print("bi_mean:", bi_mean)
display(trials_df_2[['number', 'value', 'user_attrs_bi_mean'] + [c for c in trials_df_2.columns if "params_" in c]].head(10))
# plot_param_importances(study2)
# plot_optimization_history(study2)
# plot_contour(study2, params=["learning_rate", "num_leaves"])
# plt.show()

# Top-2 Best Params Optuna

In [ ]:
top_2_params = []

for _, row in trials_df.head(2).iterrows():
    params = {col.replace("params_", ""): row[col] for col in row.index if col.startswith("params_")}
    bi_mean = row['user_attrs_bi_mean']

    top_2_params.append({
        'trial_number': row['number'],
        'params': params,
        'score': row['value'],
        'best_iter': int(bi_mean)
    })

params_tuned_21 = top_2_params[0]['params']
n_estimators_21 = top_2_params[0]['best_iter']

params_tuned_22 = top_2_params[1]['params']
n_estimators_22 = top_2_params[1]['best_iter']


In [ ]:
model_name = "tuned_21"

params_tuned_21.update({
    'n_estimators': int(1.2 * n_estimators_21),
    "device_type": "gpu",
    'verbosity': -1,
})

result_21 = era_wise_cv(
        model_name=model_name,
        params=params_tuned_21,
        train_df=train,
        )

# save_model_result(model_results, result_21)
# display(result_21['fold_results_df'])
# plot_cv_metrics(result_21['fold_results_df'], cols_fold_cv=cols_fold_cv)
# # plot_correlation_by_era(result_21['oof'], target_col=TARGET, pred_col=PREDICTION)
# # plot_prediction_target(result_21['oof'])
# # plot_feature_importance(result_21['importance_df'])
# results_cv = pd.concat([results_cv, pd.DataFrame([result_21['cv_summary']])], ignore_index=True, sort=False).round(4)
# display(results_cv)

In [ ]:
model_name = "tuned_22"

params_tuned_22.update({
    'n_estimators': int(1.2 * n_estimators_22),
    "device_type": "gpu",
    'verbosity': -1,
})

result_22 = era_wise_cv(
        model_name=model_name,
        params=params_tuned_22,
        train_df=train,
        )

# save_model_result(model_results, result_22)
# display(result_22['fold_results_df'])
# plot_cv_metrics(result_22['fold_results_df'], cols_fold_cv=cols_fold_cv)
# # plot_correlation_by_era(result_22['oof'], target_col=TARGET, pred_col=PREDICTION)
# # plot_prediction_target(result_22['oof'])
# # plot_feature_importance(result_22['importance_df'])
# results_cv = pd.concat([results_cv, pd.DataFrame([result_22['cv_summary']])], ignore_index=True, sort=False).round(4)
# display(results_cv)

In [ ]:
print("Model #1")
display(result_21['fold_results_df'])
print("Model #2")
display(result_22['fold_results_df'])
plot_cv_metrics(result_21['fold_results_df'], cols_fold_cv=cols_fold_cv, title="Model #1")
plot_cv_metrics(result_22['fold_results_df'], cols_fold_cv=cols_fold_cv, title="Model #2")

# Train/Val

In [ ]:
# model_name = "final_model"

# best_params1 = params_tuned_22


# def train_and_evaluate(train_df, val_df, feature_set, model_name, params) -> tuple:
#     start_time = time.perf_counter()

#     # Обучение модели
#     model = lgb.LGBMRegressor(**params, random_state=SEED)
#     model.fit(train_df[feature_set], train_df["target"])

#     # Прогноз на валидации
#     val_predictions = model.predict(val_df[feature_set])
#     print("Предсказание на валидации завершено")

#     # Замер времени
#     elapsed_time = time.perf_counter() - start_time
#     elapsed_str = str(timedelta(seconds=int(elapsed_time)))

#     return elapsed_str, model, val_predictions

# elapsed_str, model, val_predictions = train_and_evaluate(train, validation, feature_set, model_name, params=best_params1)
# val_predictions

In [ ]:
model_name = "final_model"

params_final = params_tuned_22


def train_validation( 
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        params: dict,
        model_name: str = "model",
        feature_set: list = FEATURES,
        target_col: str = TARGET,
        pred_col: str = PREDICTION,
        meta_col: str = 'meta_model',
    ) -> dict:

    start_time = time.perf_counter()

    # Обучение модели
    model = lgb.LGBMRegressor(**params, random_state=SEED)
    model.fit(train_df[feature_set], train_df[target_col])

    # Прогноз на validation
    val_pred = model.predict(val_df[feature_set])
    val_df = val_df.copy()
    val_df[pred_col] = val_pred

    # Время выполнения
    elapsed_time = time.perf_counter() - start_time
    elapsed_str = str(timedelta(seconds=int(elapsed_time)))

    # Вычисление метрик
    metrics_corr = compute_metrics_corr(val_df, pred_col, target_col)
    metrics_mmc = compute_metrics_mmc(val_df, pred_col,  target_col, meta_col)
    
    feature_exposure = compute_feature_exposure(
        df=val_df,
        features_df=val_df[feature_set],
        pred_col=pred_col
    )
    metrics_corr["feature_exposure"] = feature_exposure

    autocorr = compute_autocorr(val_df, pred_col)
    metrics_corr["autocorr"] = autocorr

    # Объединяем всё в один словарь метрик
    all_metrics = {
        'model_name': model_name,
        'time': elapsed_str,
        **metrics_corr,
        **metrics_mmc,
        'feature_exposure': float(feature_exposure),
        'autocorr': float(autocorr),
        '---': '---',
        **params
    }

    return {
        'model': model,
        'val_pred': val_pred,
        'val_df': val_df,
        'metrics': all_metrics,
        'elapsed_time': elapsed_str,
        'model_name': model_name
    }

result_val = train_validation(train, validation, params_final, model_name=model_name)

# plot_correlation_by_era(result_val['val_df'])
# plot_prediction_target(result_val['val_df'])

results_val = pd.concat([results_val, pd.DataFrame([result_val['metrics']])], ignore_index=True, sort=False).round(4)
display(results_val)